# Part 4 · Pandas
> 数据工程最常用的 DataFrame 操作全覆盖

## 1. 读写

In [ ]:
import pandas as pd

# --- 读 ---
df = pd.read_csv('data.csv')
df = pd.read_csv('data.csv',
    sep=',',                    # 分隔符
    header=0,                   # 表头行号（None=无表头）
    names=['a','b','c'],        # 自定义列名
    usecols=['a','c'],          # 只读指定列
    dtype={'id': str, 'amt': float},  # 指定类型（避免错误推断）
    parse_dates=['created_at'], # 自动解析为 datetime
    nrows=1000,                 # 只读前 N 行
    skiprows=[1,2],             # 跳过指定行
    na_values=['NULL','N/A',''],# 自定义 NA 标记
    encoding='utf-8',
    chunksize=10000,            # 分块读取，返回迭代器
    low_memory=False)           # 混合类型列时避免警告

# 分块读取大文件
chunks = []
for chunk in pd.read_csv('big.csv', chunksize=50000):
    chunks.append(chunk[chunk['value'] > 0])  # 每块过滤
df = pd.concat(chunks, ignore_index=True)

df = pd.read_parquet('data.parquet', columns=['id','value'])
df = pd.read_json('data.json', orient='records', lines=True)  # JSONL
df = pd.read_excel('data.xlsx', sheet_name='Sheet1')
df = pd.read_sql('SELECT * FROM orders LIMIT 100', con=engine)
df = pd.read_sql_query('SELECT ...', con=conn, params={'id': 42})
df = pd.read_feather('data.feather')    # 极快，适合临时缓存
df = pd.read_orc('data.orc')

# --- 写 ---
df.to_csv('out.csv', index=False, encoding='utf-8')
df.to_parquet('out.parquet', index=False, compression='snappy')  # 或 'gzip','zstd'
df.to_json('out.jsonl', orient='records', lines=True)  # JSONL
df.to_excel('out.xlsx', sheet_name='data', index=False)
df.to_sql('table_name', con=engine, if_exists='replace', index=False,
           method='multi', chunksize=1000)  # method='multi' 批量插入更快
df.to_feather('out.feather')

## 2. 查看 & 基本信息

In [ ]:
df.head(10)             # 前 10 行
df.tail(5)              # 后 5 行
df.sample(n=100)        # 随机抽 100 行
df.sample(frac=0.1)     # 随机抽 10%

df.shape                # (rows, cols)
df.ndim                 # 维度（2）
df.size                 # 总元素数

df.info()               # 列名、类型、非空数、内存
df.describe()           # 数值列统计（count/mean/std/min/max/分位数）
df.describe(include='all')  # 包含非数值列
df.describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99])

df.dtypes               # 每列类型
df.columns.tolist()     # 列名列表
df.index                # 行索引

df.memory_usage(deep=True)          # 每列内存（bytes）
df.memory_usage(deep=True).sum()    # 总内存

df.isnull().sum()       # 每列缺失数量
df.isnull().sum() / len(df)  # 缺失率
df.isnull().any(axis=1)     # 每行是否含缺失值

df.nunique()            # 每列唯一值数量
df['col'].unique()      # 某列的唯一值数组
df['col'].value_counts()          # 频次统计，降序
df['col'].value_counts(normalize=True)  # 比例
df['col'].value_counts(dropna=False)    # 包含 NaN

## 3. 选择 & 过滤

In [ ]:
# --- 列选择 ---
df['col']               # Series
df[['col1','col2']]     # DataFrame
df.col                  # 等价（但列名不能有空格/与方法同名）

# --- 行选择 ---
df.loc[idx]                         # 按标签
df.loc[2:5, 'name':'age']          # 行列切片（标签，含右端）
df.loc[df['age']>25, ['name','age']] # 条件行 + 特定列
df.iloc[0]                          # 第0行
df.iloc[2:5, 0:3]                  # 位置切片（不含右端）
df.at[0, 'name']                    # 单元素（标签，快）
df.iat[0, 1]                        # 单元素（位置，快）

# --- 条件过滤 ---
df[df['age'] > 25]                              # 基本条件
df[(df['age']>25) & (df['city']=='NYC')]         # 多条件 AND（用 & 不用 and）
df[(df['age']<25) | (df['city']=='LA')]          # OR
df[~df['city'].isin(['NYC','LA'])]               # NOT IN
df[df['name'].isin(['Alice','Bob'])]             # IN
df[df['score'].between(60, 100)]                 # 区间
df[df['name'].str.contains('ali', case=False)]   # 字符串包含
df[df['name'].str.startswith('A')]               # 字符串开头
df[df['col'].notna()]                            # 非空行
df[df['col'].isna()]                             # 空行

# --- query（更可读，适合动态条件）---
df.query('age > 25 and city == "NYC"')
threshold = 25
df.query('age > @threshold')         # @ 引用外部变量

# --- 类型筛选列 ---
df.select_dtypes(include='number')     # 数值列
df.select_dtypes(include='object')     # 字符串列
df.select_dtypes(exclude='datetime')   # 排除 datetime

## 4. 变换 & 新增列

In [ ]:
# --- 新增/修改列 ---
df['new_col'] = df['a'] + df['b']           # 直接赋值
df = df.assign(                              # 链式（不修改原 df）
    total = df['qty'] * df['price'],
    label = lambda x: x['total'].apply(lambda v: 'high' if v>100 else 'low')
)

# --- 重命名 ---
df.rename(columns={'old':'new', 'a':'b'}, inplace=True)
df.columns = ['col1','col2','col3']          # 整体替换
df.columns = df.columns.str.lower().str.replace(' ','_')  # 统一命名

# --- 删除列/行 ---
df.drop(columns=['col1','col2'])             # 删列
df.drop(index=[0,1,2])                       # 删行（by label）
df.drop(df[df['age']<0].index)              # 条件删行

# --- apply（按行/列执行函数）---
df['col'].apply(lambda x: x.strip())        # Series apply（逐元素）
df.apply(lambda row: row['a']+row['b'], axis=1)  # DataFrame apply 按行
df.apply(lambda col: col.max()-col.min())   # 按列（axis=0，默认）

# --- map（Series 替换/映射）---
df['grade'].map({'A':4,'B':3,'C':2})       # 值替换（无匹配→NaN）
df['grade'].map(lambda x: x.lower())       # 函数映射

# --- 类型转换 ---
df['age'] = df['age'].astype(int)
df['price'] = df['price'].astype('float32')   # 降精度省内存
df['code'] = df['code'].astype('category')    # category 类型省内存
df['flag'] = df['flag'].astype(bool)
pd.to_numeric(df['col'], errors='coerce')     # 非数值转 NaN

# --- 缺失值处理 ---
df.fillna(0)                          # 填充固定值
df.fillna(method='ffill')             # 前向填充
df.fillna(method='bfill')             # 后向填充
df.fillna(df.mean(numeric_only=True)) # 均值填充
df['col'].fillna(df['col'].median())  # 中位数填充
df.dropna()                           # 删除含缺失的行
df.dropna(subset=['name','age'])      # 只看指定列
df.dropna(thresh=3)                   # 至少有 3 个非空才保留
df.dropna(axis=1)                     # 删除含缺失的列

# --- 去重 ---
df.drop_duplicates()                          # 所有列完全一致则去重
df.drop_duplicates(subset=['id'])             # 按指定列
df.drop_duplicates(subset=['id'], keep='last') # 保留最后一条
df.duplicated()                               # bool Series

# --- 条件赋值 ---
import numpy as np
df['label'] = np.where(df['score']>=60, 'pass', 'fail')  # 简单二选一
df['grade'] = np.select(
    [df['score']>=90, df['score']>=60],
    ['A', 'B'],
    default='C'
)

## 5. GroupBy 聚合

In [ ]:
# --- 基础聚合 ---
df.groupby('city')['sales'].sum()
df.groupby(['city','month'])['sales'].agg(['sum','mean','count'])

# --- agg 多列多函数 ---
df.groupby('city').agg(
    total_sales=('sales', 'sum'),        # 命名聚合
    avg_sales=('sales', 'mean'),
    order_count=('order_id', 'nunique'),
    max_date=('date', 'max'),
    first_name=('name', 'first'),
)

# --- 自定义聚合函数 ---
df.groupby('city')['sales'].agg(lambda x: x.quantile(0.95))
df.groupby('city').agg({'sales': ['sum', lambda x: x.nlargest(3).sum()]})

# --- transform（聚合结果广播回原行，不减少行数）---
df['city_avg'] = df.groupby('city')['sales'].transform('mean')
df['pct_of_city'] = df['sales'] / df.groupby('city')['sales'].transform('sum')
df['rank_in_city'] = df.groupby('city')['sales'].rank(ascending=False)

# --- filter（过滤组）---
df.groupby('city').filter(lambda g: g['sales'].sum() > 10000)

# --- apply（复杂自定义逻辑）---
def top_n(group, n=3):
    return group.nlargest(n, 'sales')
df.groupby('city').apply(top_n, n=2)

# --- 常用参数 ---
df.groupby('city', sort=False)   # 不按 key 排序（快一些）
df.groupby('city', dropna=False) # 将 NaN 也作为一个分组

# --- 窗口聚合（滚动/扩展）---
df['rolling_avg'] = df['sales'].rolling(window=7).mean()    # 7日滚动均值
df['rolling_sum'] = df['sales'].rolling(window=7, min_periods=1).sum()
df['cumsum'] = df['sales'].expanding().sum()                 # 累计求和
df['ewm'] = df['sales'].ewm(span=7).mean()                  # 指数加权均值

## 6. 合并 & 重塑

In [ ]:
# --- merge（JOIN 操作）---
pd.merge(left, right, on='key')                        # INNER JOIN
pd.merge(left, right, on='key', how='left')            # LEFT JOIN
pd.merge(left, right, on='key', how='right')           # RIGHT JOIN
pd.merge(left, right, on='key', how='outer')           # FULL OUTER JOIN
pd.merge(left, right, left_on='l_id', right_on='r_id') # 不同列名
pd.merge(left, right, on=['key1','key2'])               # 多键
pd.merge(left, right, on='key', suffixes=('_l','_r'))  # 重名列后缀
pd.merge(left, right, on='key', validate='1:m')        # 校验关系类型
pd.merge(left, right, on='key', indicator=True)        # 添加 _merge 列标记来源

# --- concat（堆叠）---
pd.concat([df1, df2])                       # 纵向（行叠加，UNION ALL）
pd.concat([df1, df2], ignore_index=True)    # 重置行索引
pd.concat([df1, df2], axis=1)              # 横向（列合并，按索引对齐）
pd.concat([df1, df2], join='inner')        # 只保留共有列

# --- pivot_table ---
pd.pivot_table(df,
    values='sales',
    index='city',
    columns='month',
    aggfunc='sum',
    fill_value=0,
    margins=True,        # 添加合计行/列
    margins_name='Total'
)

# --- pivot（严格的一对一，无聚合）---
df.pivot(index='date', columns='metric', values='value')

# --- melt（宽转长，unpivot）---
df.melt(
    id_vars=['id','name'],          # 保留列
    value_vars=['jan','feb','mar'], # 转为行的列
    var_name='month',               # 新列名（原列名）
    value_name='sales'              # 新列名（值）
)

# --- stack / unstack ---
df.set_index(['city','month'])['sales'].unstack('month')  # 月份展开为列
wide_df.stack()                                           # 列收窄为行（多层索引）

# --- crosstab（交叉频次表）---
pd.crosstab(df['city'], df['grade'])            # 频次
pd.crosstab(df['city'], df['grade'], normalize='index')  # 行比例

## 7. 字符串 & 日期 accessor

In [ ]:
# === .str accessor（字符串列）===
df['name'].str.lower()                         # 小写
df['name'].str.upper()
df['name'].str.strip()
df['name'].str.contains('ali', case=False, na=False)  # 包含（na=False 处理空值）
df['name'].str.startswith('A')
df['name'].str.endswith('e')
df['email'].str.replace(r'@.*$', '', regex=True)     # 正则替换
df['name'].str.split(' ', expand=True)               # 分割为多列
df['name'].str.split(' ').str[0]                     # 取第一个词
df['name'].str.len()                                 # 字符串长度
df['name'].str.count('[aeiou]')                      # 正则计数
df['col'].str.extract(r'(\d{4})-(\d{2})-(\d{2})')   # 正则提取 → 多列
df['col'].str.extractall(r'(\d+)')                   # 所有匹配 → 多行
df['name'].str.get(0)                                # 取第几个字符
df['name'].str.pad(10, side='right', fillchar='-')   # 填充到固定宽度
df['col'].str.zfill(5)                               # 数字补零

# === .dt accessor（datetime 列）===
df['date'] = pd.to_datetime(df['date'])          # 先转类型
df['year']    = df['date'].dt.year
df['month']   = df['date'].dt.month
df['day']     = df['date'].dt.day
df['hour']    = df['date'].dt.hour
df['weekday'] = df['date'].dt.dayofweek          # 0=Monday
df['weekday_name'] = df['date'].dt.day_name()   # 'Monday'...
df['quarter'] = df['date'].dt.quarter
df['date_only'] = df['date'].dt.date            # 转为 date 对象
df['date'].dt.strftime('%Y-%m')                 # 格式化字符串
df['date'].dt.floor('H')                        # 向下取整到小时
df['date'].dt.ceil('D')                         # 向上取整到天
df['date'].dt.tz_localize('UTC')                # 设置时区
df['date'].dt.tz_convert('America/New_York')    # 转换时区
df['date'].dt.to_period('M')                    # 转为 Period（月）

# 日期计算
df['days_ago'] = (pd.Timestamp.now() - df['date']).dt.days

# pd.to_datetime
pd.to_datetime('2024-01-15')
pd.to_datetime(['2024-01-15','2024-01-16'])
pd.to_datetime(df['col'], format='%d/%m/%Y', errors='coerce')  # 指定格式，错误→NaT

# pd.date_range
pd.date_range('2024-01-01', '2024-12-31', freq='D')  # 每天
pd.date_range('2024-01-01', periods=12, freq='MS')   # 每月第一天，12个
pd.date_range('2024-01-01', periods=52, freq='W')    # 每周

## 8. 排序 & 排名 & 统计

In [ ]:
# --- 排序 ---
df.sort_values('sales', ascending=False)              # 降序
df.sort_values(['city','sales'], ascending=[True,False])  # 多列
df.sort_values('name', na_position='last')           # NaN 放末尾
df.sort_index()                                       # 按索引排序

# --- 排名 ---
df['rank'] = df['sales'].rank(ascending=False)
df['rank'] = df['sales'].rank(method='dense')        # DENSE_RANK
df['rank'] = df['sales'].rank(method='min')          # MIN RANK
df['rank'] = df['sales'].rank(pct=True)              # 百分位排名

# --- Top N ---
df.nlargest(5, 'sales')                               # 最大5条（快，不需要全排序）
df.nsmallest(5, 'sales')
df.nlargest(5, ['sales','profit'])                    # 多列排序取 Top N

# --- 统计 ---
df['col'].mean()
df['col'].median()
df['col'].std()
df['col'].var()
df['col'].quantile([0.25, 0.5, 0.75])               # 分位数
df['col'].skew()                                     # 偏度
df['col'].kurt()                                     # 峰度
df['col'].corr(df['col2'])                           # 两列相关系数
df.corr()                                            # 相关矩阵

# --- 分箱 ---
pd.cut(df['age'], bins=[0,18,35,60,100],
       labels=['teen','young','middle','senior'])     # 等宽分箱
pd.qcut(df['income'], q=4, labels=['Q1','Q2','Q3','Q4'])  # 等频分箱（分位数）

# --- resample（时序聚合）---
df.set_index('date').resample('W')['sales'].sum()    # 按周聚合
df.set_index('date').resample('M')['sales'].agg(['sum','mean'])  # 按月
df.set_index('date').resample('Q-DEC').last()        # 季度末

## 9. 性能优化技巧

In [ ]:
# 1. 降低数据类型精度
df['id'] = df['id'].astype('int32')         # int64 → int32
df['score'] = df['score'].astype('float32') # float64 → float32
df['city'] = df['city'].astype('category')  # 低基数字符串 → category

# 2. 向量化 > apply > for loop
# ❌ 慢：逐行 apply
df['total'] = df.apply(lambda r: r['qty'] * r['price'], axis=1)
# ✅ 快：向量化
df['total'] = df['qty'] * df['price']

# 3. query/eval 避免临时对象
df.eval('total = qty * price', inplace=True)  # 快于赋值
df.query('age > 25')                           # 有时比 boolean indexing 快

# 4. inplace=True 原地修改（节省复制，但有时没更快，视情况）
df.fillna(0, inplace=True)

# 5. 读 parquet 只加载需要的列
df = pd.read_parquet('data.parquet', columns=['id','sales','date'])

# 6. 读 csv 分块处理
result = pd.concat(
    [chunk.groupby('city')['sales'].sum()
     for chunk in pd.read_csv('big.csv', chunksize=100_000)]
).groupby(level=0).sum()

# 7. 检查内存使用
def mem_usage(df):
    mem = df.memory_usage(deep=True).sum()
    return f'{mem / 1024**2:.2f} MB'
print(mem_usage(df))